# Time-Series Analysis: Algorithmic Response to AIGC Supply Expansion
---
**Context:** This notebook executes the time-series regression framework required to investigate the *Scale-over-Preference* dynamics. It evaluates the dynamic temporal relationships between the AIGC/HGC content supply scale and the platform's algorithmic exposure allocation. 

The analysis follows a two-step procedure:
1.  **Stationarity Verification:** Conducting Augmented Dickey-Fuller (ADF) and Kwiatkowski-Phillips-Schmidt-Shin (KPSS) unit root tests on the first-differenced series (reproducing Supplementary Table 10).
2.  **Granger Causality Testing:** Implementing a robust block-bootstrapped Granger causality test across multiple lags to determine directional precedence (reproducing Supplementary Table 11).

**Variables Analyzed:**
- `supply_scale_aigc` / `supply_scale_hgc`: The volume of daily content uploads.
- `exposure_median_aigc` / `exposure_median_hgc`: The algorithmic impressions allocated.

### Step 1: Data Preprocessing & Exogenous Shock Controls
Here we ingest the daily panel dataset. To ensure robust econometric estimations, we explicitly account for structural breaks by flagging statutory holidays (which alter user consumption patterns) and calculating the first-differences of our focal variables to guarantee stationarity.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.append('../core')
from regression import bootstrap_granger_test, bootstrap_parameter_ci, find_optimal_lag_kneedle

# 1. Load the Panel Data
df = pd.read_csv('daily_panel.csv')
df['date'] = pd.to_datetime(df['date'])

# 2. Control for Temporal Heterogeneity (Statutory Holidays)
holiday_dates = [
    '2024-06-08', '2024-06-09', '2024-06-10', '2024-09-15', '2024-09-16', '2024-09-17',
    '2024-10-01', '2024-10-02', '2024-10-03', '2024-10-04', '2024-10-05', '2024-10-06',
    '2024-10-07', '2025-01-01', '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31',
    '2025-02-01', '2025-02-02', '2025-02-03', '2025-04-04', '2025-04-05', '2025-04-06',
    '2025-05-01', '2025-05-02', '2025-05-03', '2025-05-04', '2025-05-05'
]
holiday_dates = [pd.to_datetime(date) for date in holiday_dates]
df['holiday'] = df['date'].isin(holiday_dates).astype(int)
df['holiday_entry'] = ((df['holiday'] == 1) & (df['holiday'].shift(1) == 0)).astype(int)
df['holiday_exit'] = ((df['holiday'] == 0) & (df['holiday'].shift(1) == 1)).astype(int)

# 3. First Differencing of Core Variables
# Mapping from raw system variables to paper-defined metrics
vars_to_diff = ['exp31_p50_aigc', 'exp31_p50_hgc', 'exp1_p50_aigc', 'exp1_p50_hgc', 'scale_aigc', 'scale_hgc']
df_diff = df[vars_to_diff].diff().dropna().reset_index(drop=True)

# 4. Generate Exogenous Dummies
df['wd'] = df['date'].dt.weekday
wd_dummies = pd.get_dummies(df['wd'], prefix='wd', drop_first=True).astype(int).iloc[1:].reset_index(drop=True)
holiday_dummies = df[['holiday_entry', 'holiday_exit']].iloc[1:].reset_index(drop=True)
exog_combined = pd.concat([wd_dummies, holiday_dummies], axis=1)

data_diff = pd.concat([df_diff, exog_combined], axis=1)
print("✅ Data preprocessing and first-differencing completed successfully.")

### Step 2: Stationarity and Unit Root Tests (Supplementary Table 10)
To preclude the risk of spurious regression, we perform rigorous unit root testing on the first-differenced variables using both the Augmented Dickey-Fuller (ADF) and the Kwiatkowski-Phillips-Schmidt-Shin (KPSS) tests. 
- **ADF Test**: Rejection of the null hypothesis ($p < 0.05$) indicates stationarity.
- **KPSS Test**: Failure to reject the null hypothesis ($p > 0.05$) indicates stationarity.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings('ignore')

def run_unit_root_tests(df_diff, vars_dict):
    results = []
    for col, name in vars_dict.items():
        series = df_diff[col].dropna()
        
        # ADF Test Assessment
        adf_stat, adf_p, _, _, _, _ = adfuller(series, autolag='AIC')
        adf_sig = '***' if adf_p < 0.001 else '**' if adf_p < 0.01 else '*' if adf_p < 0.05 else ''
        
        # KPSS Test Assessment
        kpss_stat, kpss_p, _, _ = kpss(series, regression='c', nlags='auto')
        
        results.append({
            'Variables (First Difference)': name,
            'ADF Test Statistic': f"{adf_stat:.2f} ({adf_p:.3f}){adf_sig}",
            'KPSS Test Statistic': f"{kpss_stat:.2f} ({kpss_p:.3f})"
        })
    return pd.DataFrame(results)

# Rename internal columns to academic variable names
vars_dict = {
    'exp31_p50_aigc': 'Median AIGC Exposure (31-day)',
    'exp31_p50_hgc': 'Median HGC Exposure (31-day)',
    'exp1_p50_aigc': 'Median AIGC Exposure (1-day)',
    'exp1_p50_hgc': 'Median HGC Exposure (1-day)',
    'scale_aigc': 'AIGC Supply Scale',
    'scale_hgc': 'HGC Supply Scale'
}

stationarity_table = run_unit_root_tests(df_diff, vars_dict)
print("=== Supplementary Table 10: Unit root tests for variables in first differences ===")
display(stationarity_table)

### Step 3: Block-Bootstrapped Granger Causality Analysis (Supplementary Table 11)
Having satisfied the stationarity prerequisite, we evaluate whether an expansion in content supply triggers algorithmic exposure adjustments. We utilize a joint F-test framework reinforced by block bootstrapping to accommodate potential heteroskedasticity and serial correlation, testing across lags $p \in [1, 6]$.

In [ ]:
B_ITERATIONS = 2000
BLOCK_SIZE = 7
MAX_LAG = 6

print("Executing Granger Causality: AIGC Supply Expansion -> Algorithmic Exposure Allocation (1d)")
print("-" * 80)

aigc_causality_results = {}
for lag in range(1, MAX_LAG + 1):
    res = bootstrap_granger_test(
        target_exposure=data_diff['exp1_p50_aigc'].values,
        focal_supply=data_diff['scale_aigc'].values,
        control_supply=data_diff['scale_hgc'].values,
        max_lags=lag, 
        exog_vars=exog_combined,
        block_size=BLOCK_SIZE, 
        B_iterations=B_ITERATIONS, 
        random_state=42
    )
    aigc_causality_results[f'lag_{lag}'] = res
    
    # Compute Confidence Intervals for the supply elasticity parameters
    ci_res = bootstrap_parameter_ci(
        target_exposure=data_diff['exp1_p50_aigc'].values,
        focal_supply=data_diff['scale_aigc'].values,
        control_supply=data_diff['scale_hgc'].values,
        max_lags=lag, 
        exog_vars=exog_combined,
        block_size=BLOCK_SIZE, 
        B_iterations=B_ITERATIONS, 
        alpha=0.05, 
        random_state=42
    )
    
    sig_star = '***' if res['p_boot'] < 0.001 else '**' if res['p_boot'] < 0.01 else '*' if res['p_boot'] < 0.05 else 'ns'
    print(f"Lag {lag}: F-Stat = {res['F_original']:.3f} | Bootstrapped p-value = {res['p_boot']:.4f} ({sig_star})")

print("\nExecuting optimal lag detection via Kneedle algorithm (AIC Minimization)...")
opt_lag, _, _ = find_optimal_lag_kneedle(
    target_exposure=data_diff['exp1_p50_aigc'].values,
    focal_supply=data_diff['scale_aigc'].values,
    control_supply=data_diff['scale_hgc'].values,
    max_lag=MAX_LAG, 
    exog_vars=exog_combined
)
print(f"Optimal structural lag selected: {opt_lag}")